# 🏦 Pandas for auditors — Exercises Level 2: Medium

**Context**: You are an auditor in charge of reviewing the **residential real-estate loan portfolio** of a regional bank.
The accounting department has sent you an export of all the **installments** of the first half of 2024:
for each loan, you have the amount due, the amount actually collected, the payment status and the number of days overdue.

**Objective**: identify the branches and loan types at risk, measure exposure to unpaid amounts,
and check data quality.

**Skills covered**: multi-condition filtering (`&`, `|`, `~`, `.isin()`, `.between()`),
computed columns (`np.where`), `groupby` + `.agg()`, `pivot_table`, data quality.

> ℹ️ The data is entirely fictional and randomly generated.

## 0. Data generation — run this first

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
np.random.seed(2025)

n = 350

branches = ['North Branch', 'South Branch', 'East Branch', 'West Branch', 'Central Branch']
loan_types = ['Residential Real Estate', 'Commercial Real Estate', 'Buy-to-Let']
statuses = ['Paid', 'Minor Delay', 'Severe Delay', 'Unpaid']

payment_status = np.random.choice(statuses, n, p=[0.68, 0.15, 0.10, 0.07])
due_amounts = np.round(np.random.uniform(500, 4500, n), 2)
# paid amount: 100% if paid, partial otherwise
payment_ratio = np.where(
    payment_status == 'Paid', 1.0,
    np.where(payment_status == 'Minor Delay',
             np.random.uniform(0.0, 1.0, n),
             np.where(payment_status == 'Severe Delay',
                      np.random.uniform(0.0, 0.5, n),
                      0.0))
)
paid_amounts = np.round(due_amounts * payment_ratio, 2)

days_late = np.where(
    payment_status == 'Paid', 0,
    np.where(payment_status == 'Minor Delay', np.random.randint(1, 30, n),
    np.where(payment_status == 'Severe Delay', np.random.randint(30, 90, n),
             np.random.randint(90, 365, n)))
)

loans = pd.DataFrame({
    'loan_id':          [f'PR{str(i).zfill(5)}' for i in range(1, n + 1)],
    'client_id':        np.random.randint(50000, 51000, n),
    'branch':           np.random.choice(branches, n),
    'loan_type':        np.random.choice(loan_types, n, p=[0.55, 0.25, 0.20]),
    'interest_rate':    np.round(np.random.uniform(1.5, 4.5, n), 2),
    'due_date':         pd.to_datetime('2024-01-01') + pd.to_timedelta(
                            np.random.randint(0, 180, n), unit='D'),
    'due_amount':       due_amounts,
    'paid_amount':      paid_amounts,
    'payment_status':   payment_status,
    'days_overdue':     days_late,
})

# intentional missing values
loans.loc[np.random.choice(loans.index, 10, replace=False), 'interest_rate'] = np.nan
loans.loc[np.random.choice(loans.index, 5,  replace=False), 'paid_amount'] = np.nan

# duplicates (same loan entered twice)
dups = loans.sample(4, random_state=99).copy()
dups['loan_id'] = [f'DUP{i}' for i in range(4)]
loans = pd.concat([loans, dups], ignore_index=True)
loans = loans.sample(frac=1, random_state=3).reset_index(drop=True)

print('Loans dataset ready:', loans.shape[0], 'installments,', loans.shape[1], 'columns')
loans.head()

**Column description**

| Column | Description |
|---|---|
| `loan_id` | loan identifier |
| `client_id` | client identifier |
| `branch` | managing branch |
| `loan_type` | loan category |
| `interest_rate` | annual rate (%) |
| `due_date` | installment due date |
| `due_amount` | amount due (€) |
| `paid_amount` | amount actually collected (€) |
| `payment_status` | Paid / Minor Delay / Severe Delay / Unpaid |
| `days_overdue` | number of days overdue (0 if Paid) |

---
## Exercise 1 — Multi-condition filtering

**Questions:**
1. List the installments that are **`Severe Delay`** or **`Unpaid`** — how many are there?
2. Among the `Unpaid` ones, which have an **amount due over €3,000**?
3. List the installments whose delay is **between 30 and 90 days** (use `.between()`).
4. List the loans of type `Commercial Real Estate` OR `Buy-to-Let`
   whose status is **not** `Paid` (use `.isin()` and `~`).

In [ ]:
# 1. Severe Delay or Unpaid
# Your code here


In [ ]:
# 2. Unpaid with amount > 3,000
# Your code here


In [ ]:
# 3. Delay between 30 and 90 days
# Your code here


In [ ]:
# 4. Commercial/Buy-to-Let loans NOT paid
# Your code here


---
## Exercise 2 — Computed columns

**Questions:**
1. Create a column **`unpaid_amount`** = `due_amount` − `paid_amount`
   (treat `NaN` values of `paid_amount` as if the paid amount were 0 — use `.fillna(0)`).
2. Create a boolean column **`critical_delay`** that is `True` if `days_overdue >= 60`
   (use a direct comparison).
3. Create a column **`delay_category`** with three values:
   - `'None'` if `days_overdue == 0`
   - `'Minor'` if `days_overdue` is between 1 and 59
   - `'Critical'` if `days_overdue >= 60`

   (Tip: `np.select([condition1, condition2], [choice1, choice2], default=choice3)`)

In [ ]:
# 1. Unpaid amount
# Your code here


In [ ]:
# 2. Critical delay (boolean)
# Your code here


In [ ]:
# 3. Delay category (np.select)
# Your code here


---
## Exercise 3 — Summaries with `groupby`

**Questions:**
1. For each **branch**, compute: the total amount due, the total amount paid,
   and the total unpaid amount. Sort by unpaid amount descending.
   Which branch has the highest exposure?
2. For each **loan type**, compute: the number of installments, the average delay,
   and the total unpaid amount.
3. For each **branch**, compute the **number** of installments per `payment_status`
   (groupby on two columns).

In [ ]:
# 1. Summary by branch
# Your code here


In [ ]:
# 2. Summary by loan type
# Your code here


In [ ]:
# 3. Number of installments by branch and status
# Your code here


---
## Exercise 4 — Pivot table (`pivot_table`)

**Questions:**
1. Create a pivot table with **branches as rows**, **loan types as columns**,
   and the **sum of the unpaid amount** in the cells.
2. Create a second pivot table with **branches as rows**, **statuses as columns**,
   and the **number of installments** in the cells.
   (Tip: `aggfunc='count'`, `values='loan_id'`)

In [ ]:
# 1. Pivot: unpaid amount by branch × loan type
# Your code here


In [ ]:
# 2. Pivot: number of installments by branch × status
# Your code here


---
## Exercise 5 — Data quality

**Questions:**
1. How many **missing values** are there per column?
2. Identify the **duplicate** rows based on the columns `client_id`,
   `due_date`, `due_amount`. How many rows are affected?
3. Create a **cleaned** version of the DataFrame: without duplicates (keep the first occurrence)
   and with `NaN` values of `paid_amount` replaced by 0.
   How many rows remain?

In [ ]:
# 1. Missing values per column
# Your code here


In [ ]:
# 2. Business duplicates
# Your code here


In [ ]:
# 3. Cleaned DataFrame
# Your code here


---
## Exercise 6 — Summary analysis: branches at risk

The audit committee asks you for a **ranking of branches** by their unpaid rate.

**Build a table with, for each branch:**
- The total number of installments
- The number of installments in `Severe Delay` or `Unpaid`
- The **default rate** = (severe delays + unpaid) / total, as %
- The total unpaid amount

Sort by default rate descending.

(Tip: compute the aggregates separately via `groupby`, then combine with `merge` or create the columns one by one.)

In [ ]:
# Branches at risk analysis
# Your code here
